## <a name="traffic_flow_collection">交通流量检测</a>

我们已经介绍了两种不同的目标检测模型。在交通流量检测场景下，Region-free的YoloV3模型速度较快，但是精度稍差。我们在判定视频中经过的车辆时，可以按照车牌号码作为车辆的唯一标识，将一段时间内视频中出现的车牌号码放入集合，这样，即可统计出一段时间内经过的车辆数量。因此，我们可以使用速度较快的YoloV3对车辆进行快速检测，然后将车辆图像存入队列，交给OCR模型对车牌号码进行识别。在这一章节中，我们综合使用YoloV3和OCR模型，共同完成交通流量的检测。

### 视频帧处理

我们训练的YoloV3的输入一系列图片，每个图片可以看成三位数组，即 $(width, height, channels)$，我们在进行预测时，为了更好的利用现代硬件的优势，每次传入一批(batch)数据，因此作为输入的一系列图片，可以看成一个四维数组，即$(num_images, width, height, channels)$。而视频恰好是一个帧序列，每一帧为一张图片。因此，我们先将视频文件进行处理，提取出帧序列，再传入YoloV3模型进行目标检测。

我们选择OpenCV对视频进行处理：

In [ ]:
from time import time
import cv2
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

video_filepath = './data/street_traffic.mp4'
cap = cv2.VideoCapture(video_filepath)

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1
print('{} frames in the video'.format(frame_count))
count = 0

nums_to_cap = 4 if frame_count > 4 else frame_count
for i in range(nums_to_cap):
    _, frame = cap.read()
    rgb_img_array = frame[:, :, ::-1] # OpenCV默认为BGR模式，这里改为RGB模式，以正常显示图片
    plt.subplot(2, 2, i+1)
    plt.axis('off')
    plt.title('frame{}'.format(i))
    plt.imshow(rgb_img_array)
cap.release()
plt.show()


# 提取所有视频帧
# while cap.isOpened():
#     # Extract the frame
#     ret, frame = cap.read()
#     # Write the results back to output location.
#     cv2.imwrite("./data/video_frames/%#05d.jpg" % (count+1), frame)
#     count = count + 1
#     # If there are no more frames left
#     if (count > (frame_count-1)):
#         # Log the time again
#         time_end = time()
#         # Release the feed
#         cap.release()
#         # Print stats
#         print ("Done extracting frames.\n%d frames extracted" % count)
#         print ("It took %d seconds forconversion." % (time_end-time_start))
#         break

通过OpenCV，我们就把视频流中的帧提取出来，获得每一帧的图像。frame的格式为np.ndarray，恰好可以作为三维数组直接输入给Yolo模型。需要注意的是，OpenCV提取的像素数组的颜色通道顺序是BGR，因此，我们一般通过Python数组的下标选择机制，使用[:, :, ::-1]将像素排列为RGB顺序。其中前两个`:`表示提取该维度下所有数据，最后一个`::-1`表示提取该维度下所有数据并逆转排列顺序，这样得到的像素恰好是按照RGB排列。

### 使用YoloV3 和 OCR模型统计车牌信息

接下来，我们使用YoloV3和OCR模型，将OpenCV提取得到的图像信息中的车牌识别出来。首先，我们准备YoloV3模型：

In [ ]:
## 准备YoloV3先验Anchor
import numpy as np
anchor_path = "./data/yolov3_anchors.txt"
with open(anchor_path) as f:
    anchors = f.readline()
anchors = [float(x) for x in anchors.split(',')]
anchors = np.array(anchors).reshape(-1, 2)

## 加载YoloV3模型文件
from src.yolo3.model import yolo_body
from keras.layers import Input
yolo_weights_path = './data/yolov3.h5'
yolo_model = yolo_body(Input(shape=(None,None,3)), len(anchors)//3, 80)
yolo_model.load_weights(yolo_weights_path)

我们的OCR模型分为两个部分，一个是EAST模型，用于文字区域的检测；另一个是CRNN模型，用于字符的识别。

In [ ]:
## 加载EAST模型
from PIL import Image
import cv2
from keras.models import load_model, model_from_json
import os
import time
import numpy as np
import tensorflow as tf
from src.ocr.east.eval import resize_image, detect, sort_poly
model_path = './data/east.h5'
json_path = './data/model.json'

with open(json_path,'r') as f:
    loaded_model_json = f.read()
# 读取模型结构
east_model = model_from_json(loaded_model_json, custom_objects={'tf': tf, 'RESIZE_FACTOR': 2})
# 读取权重文件
east_model.load_weights(model_path)

In [ ]:
import string

from src.ocr.crnn.models import CRNN_STN

class CRNNConfig:
    def __init__(self):
        self.characters = '0123456789'+string.ascii_lowercase
        self.width = 200
        self.height = 31
        self.nb_channels = 1
        self.lstm_nb_units = [128, 128]
        self.conv_filter_size = [64, 128, 256, 256, 512, 512, 512]
        self.model_path = 'data/crnn.h5'
        self.dropout_rate = 0.25
        self.label_len = 16

cfg = CRNNConfig()
_, crnn_model = CRNN_STN(cfg)    
crnn_model.load_weights(cfg.model_path)

所有模型加载完毕，我们处理视频帧时，顺序为：

video $\to$ frame $\to$ Yolo $\to$ EAST $\to$ CRNN $\to$ 车牌号码

### 图像处理技巧

原始视频文件的分辨率较高(720p或更高)，Yolo模型处理分辨率较高的图片会耗费较长时间，而对OCR模型而言，图片尺寸变小，分辨率过低会导致识别精度下降。因此，在处理视频帧时，我们先将原始图像尺寸缩小，得到一个副本，再由YoloV3对尺寸较小的副本进行识别，得到车辆的包围框。这时，得到的目标可能是远处的车辆，是无法辨识出车牌信息的，我们通过一个目标面积与原图面积比例的阈值，对过小的目标进行过滤，只保留较大的目标，然后将目标的区域映射到原图中，从原图中取出原始大小的图像，再传给OCR模型进行文字识别，以提高识别精度。

## <p style="color=red;font-family:Consolas">TODO 添加示意图，修正文字表述</p>

接下来，我们提取视频中的第一帧，进行示范。

In [ ]:
cap = cv2.VideoCapture(video_filepath)
height = int(frame.shape[0]/2)
width = int(frame.shape[1]/2)

new_image_size = (width, height)

_, frame = cap.read()
resized_frame = cv2.resize(frame, dsize=new_image_size) # 无需差值
# res = cv2.resize(img, dsize=(54, 140), interpolation=cv2.INTER_CUBIC)
cap.release()

## 注意，OpenCV提取的像素按照BGR通道顺序排列
image = Image.fromarray(frame[:, :, ::-1])
image

得到图像后，我们用YoloV3模型检测车辆：

In [ ]:
from src.yolo3.utils import letterbox_image
# resize input image
new_image_size = (image.width - (image.width % 32), image.height - (image.height % 32))
boxed_image = letterbox_image(image, new_image_size)
image_data = np.array(boxed_image, dtype='float32')

image_data /= 255.
image_data = np.expand_dims(image_data, 0)
image_data.shape

In [ ]:
import keras.backend as K
sess = K.get_session()

from src.yolo3.model import yolo_eval

input_image_shape = K.placeholder(shape=(2, ))
iou = 0.45
score = 0.3

boxes, scores, classes = yolo_eval(
    yolo_model.output, 
    anchors,
    80, 
    input_image_shape,
    score_threshold=score, 
    iou_threshold=iou)

print(boxes)
print(scores)
print(classes)

out_boxes, out_scores, out_classes = sess.run(
    [boxes, scores, classes],
    feed_dict={
        yolo_model.input: image_data,
        input_image_shape: [image.size[1], image.size[0]],
        K.learning_phase(): 0
    })

In [ ]:
regions = []

image_area = image.size[0] * image.size[1]
REGION_THRESHOLD = 0.05

# Yolo模型中车辆相关的分类id：
# {"id": 2, "name": "car"},
# {"id": 5, "name": "bus"},
# {"id": 7, "name": "truck"},

vichel_class_ids = [2, 5, 7]

def is_big_enough(region, threshold):
    w, h = region.size
    return w*h / image_area >= threshold

predict_classes = []

for i, c in reversed(list(enumerate(out_classes))):
    predicted_class = c
    box = out_boxes[i]
    score = out_scores[i]

    top, left, bottom, right = box
    top = max(0, np.floor(top + 0.5).astype('int32'))
    left = max(0, np.floor(left + 0.5).astype('int32'))
    bottom = min(image.size[1], np.floor(bottom + 0.5).astype('int32'))
    right = min(image.size[0], np.floor(right + 0.5).astype('int32'))
    print(predicted_class, score, (left, top), (right, bottom))
    region = image.crop((left, top, right, bottom))
    
    if predicted_class in vichel_class_ids and is_big_enough(region, REGION_THRESHOLD):
        regions.append(region)

In [ ]:
num_regions = len(regions)
print('{} big regions'.format(num_regions))

def get_rows_cols(n):
    rows = int(n // 2)
    temp = n / rows
    cols = int(temp)+1 if temp > int(temp) else int(temp)
    return rows, cols

rows, cols = get_rows_cols(num_regions)

plt.figure(figsize=(20, 10))
for i in range(num_regions):
    plt.subplot(rows, cols, i+1)
    plt.imshow(regions[i])
    plt.axis('off')
plt.show()

### OCR检测车牌信息

首先，我们将检测到的足够大的车辆图像传给EAST模型，识别其中的文字区域

In [ ]:
detect_region = regions[0]
img = np.array(regions[0])

# 图片尺寸调整
img_resized, (ratio_h, ratio_w) = resize_image(img)
img_resized = (img_resized / 127.5) - 1

plt.axis('off')
plt.imshow(img);

In [ ]:
# 对图片进行文本区域预测，完成FCN部分
score_map, geo_map = east_model.predict(img_resized[np.newaxis, :, :, :])
# 对预测结果进行NMS计算，完成NMS部分
boxes, timer = detect(score_map=score_map, geo_map=geo_map)

In [ ]:
img_to_draw = img.copy()
text_areas = []
if boxes is not None:
    boxes = boxes[:, :8].reshape((-1, 4, 2))
    boxes[:, :, 0] /= ratio_w
    boxes[:, :, 1] /= ratio_h
for box in boxes:
    # to avoid submitting errors
    box = sort_poly(box.astype(np.int32))
    if np.linalg.norm(box[0] - box[1]) < 5 or np.linalg.norm(box[3] - box[0]) < 5:
        continue
    # cv2 BGR格式的图片，需要将最后一个维度(颜色通道)顺序逆转
    # cv2.polylines(img[:, :, ::-1], [box.astype(np.int32).reshape((-1, 1, 2))], True, color=(0, 255, 0),thickness=2)
    
    # 我们已经统一使用RGB格式，这里无需反转img的颜色通道
    # cv2.polylines(img_to_draw, [box.astype(np.int32).reshape((-1, 1, 2))], True, color=(0, 255, 0),thickness=2)

    left = np.min(box[:, 0])
    right = np.max(box[:, 0])
        
    top = np.min(box[:, 1])
    bottom = np.max(box[:, 1])
    text_area = img_to_draw[top: bottom, left: right]
    text_areas.append(text_area)

In [ ]:
rows, cols = get_rows_cols(len(text_areas))

for i in range(len(text_areas)):
    plt.subplot(rows, cols, i+1)
    plt.title('text_area{}'.format(i))
    plt.axis('off')
    plt.imshow(text_areas[i])
plt.show()

接下来,我们使用CRNN模型对字符进行预测

In [ ]:
from src.ocr.crnn.utils import preprocess_image

def predict_text(model, img):
    y_pred = model.predict(img[np.newaxis, :, :, :])
    print(y_pred.shape) # y_pred holds the possibilities of a char for each area
    shape = y_pred[:, 2:, :].shape
    ctc_decode = K.ctc_decode(y_pred[:, 2:, :], input_length=np.ones(shape[0])*shape[1])[0][0]
    ctc_out = K.get_value(ctc_decode)[:, :cfg.label_len]
    result_str = ''.join([cfg.characters[c] for c in ctc_out[0]])
    result_str = result_str.replace('-', '')
    return result_str

rows, cols = get_rows_cols(len(text_areas))
for i in range(len(text_areas)):
    text_area = text_areas[i]
    gray_image = cv2.cvtColor(text_area, cv2.COLOR_BGR2GRAY)
    processed_area = preprocess_image(gray_image, cfg)
    result = predict_text(crnn_model, processed_area)
    
    plt.subplot(rows, cols, i+1)
    plt.title('result: {}'.format(result))
    plt.axis('off')
    plt.imshow(text_area)
plt.show()

在预测函数predict_text中,我们可以看到，预测的原始结果y_pred为1xNxM的ndarray。N为文字区域的个数，M为字符分类的个数，数组元素为每个区域是单个字符的概率。之后,我们对每个文字区域的M个概率取最大值对应的字符,并对分类相同的文字区域进行合并,得到最终预测的文本结果。

需要指出,我们采用的CRNN模型是针对字母和数字进行训练得到的,在此只做为一个示例。在真正的国内交通场景下,需要对模型进行迁移学习训练,使模型能够识别中国车牌中省份的简称,并按照一定的车牌规则对识别的字符串进行筛选,以统计当前时刻下真正的车牌信息。

### 软件工程化

在上面的代码中，我们演示了数据从视频流到图像再到文字的处理过程。在实际项目中，我们需要利用工程化的手段，将不同的逻辑划分成功能独立的模块，模块之间互相调用，共同完成整个流程。单个独立的功能模块也容易进行集群化，可以有效的利用硬件资源，提高软件服务的稳定性和性能。

接下来，我们将处理流程分成视频处理模块，区域检查模块和字符识别模块，并结合多线程和异步编程手段，构建一个整体的视频OCR处理流程。其中，各个模块的功能规划如下：

- 视频处理模块：接受视频片段，处理成帧序列并传递给区域检查模块

- 区域检查模块：使用YoloV3和EAST模型，提取视频帧中的文字区域，并传给字符识别模块

- 字符识别模块：使用CRNN模型，识别出文字区域中的字符，并存储数据

所有的模块代码均存放在`src/project/`目录下，每个模块以web服务的形式提供API，基本的业务逻辑一致。我们首先查看`src/project/video_processing_service.py`，该模块提取视频帧：

In [ ]:
!cat src/project/video_processing_service.py

假设，我们有一个摄像头定期将一定时间范围的视频片段发送给该服务，要求请求的http header中有time_range表示时间范围。`video_processing_service`收到视频文件后，将文件保存到本地，并启动一个新的线程，用OpenCV提取视频帧，然后再将每一帧图像传递给下一个文字区域提取的服务。下面，我们来看一下文字提取和字符识别服务的代码：

In [ ]:
!cat src/project/region_detect_service.py

在`region_detect_service`模块中，我们将视频帧中的车辆提取出来，并检测车辆图像中的文字区域，然后通过OpenCV将文字区域编码为字节流，传递给字符识别模块。

In [ ]:
!cat src/project/ocr_service.py

字符识别模块得到文字区域的字节流，将其解码得到图像信息，然后使用CRNN模型检测文字。这里我们没有给出具体的文字处理逻辑，我们不妨设想这样一个简单直接的逻辑：得到文字后，按照车牌的规格进行过滤，最后，将时间范围内的车牌保存为一个集合，确保车牌不重复，这样，该集合的元素个数，就是这段时间内通过的车辆数量。统计完车辆数量后，我们就可以按照PeMS数据集的格式保存流量数据了。

可以看到，所有服务的逻辑基本一致，在接收到web请求后，启动一个新的线程处理业务逻辑，不阻塞请求。在执行业务逻辑时，需要对所有异常情况进行处理或者记录。time_range在每个服务中都进行了保留，这样，每个模块得到的数据都可以关联time_range，我们收集、生成的数据越详细，后续处理数据时，就可以更加灵活的组合数据的形式。

这里给出的是一个极简的工程化示例，从AI模型到实际的生产项目，我们还要考虑异常处理、数据存储、服务部署等很多问题。

<p>
<font size=3 color=red>
    <ul>
        <li>扩展1：project/目录下有一个video_camera.py文件，在这个文件中，使用OpenCV读取示例交通视频，将其分成10秒的视频片段，并上传给video_processing_service模块。
        <li>扩展2：在单个GPU的环境下，多个模型的加载和运行可能导致显存不足而产生OOM问题。请对Keras进行配置实现显存的按需分配。
        <li>扩展3：如果环境允许，使用3台服务器，部署上述3种服务，使之互相调用，完成从视频到车牌提取的流程。
    </ul>
</font>
</p>